# actas-cnn — Preprocesamiento (Colab)

**Que hace:** baja las actas (PDFs) + labels de Hugging Face, las renderiza, detecta los digitos, recorta y etiqueta, arma los manifests y **publica el bundle de crops en HF** para que el notebook entregable lo consuma.

**Esta es la superficie que mas se itera:** para cambiar *como se detectan los digitos*, edita la celda marcada `PREPROCESAMIENTO` y vuelve a correr todo.

Requiere un `HF_TOKEN` con permiso de escritura (panel de secretos de Colab) para la subida final.

## 0. Setup

In [ ]:
# Dependencias (Colab ya trae torch, torchvision, numpy, pandas, matplotlib)
%pip install -q pymupdf==1.27.2.3 opencv-python-headless huggingface_hub pyarrow
print("deps instaladas")

In [ ]:
# === Config + entorno ===
import os, tarfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

def torch_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
DEVICE = torch_device(); print("device:", DEVICE)
if DEVICE.type != "cuda":
    print("AVISO: sin GPU CUDA. En Colab: Runtime -> Change runtime type -> T4 GPU. "
          "Sin GPU el entrenamiento es MUY lento.")

HF_DATASET_REPO = "f3r21/actas-cnn-dataset"   # PDFs + labels (+ crops_bundle si 01 ya corrio)
HF_MODEL_REPO   = "f3r21/actas-cnn-model"     # checkpoints

WORK = Path("/content") if Path("/content").exists() else Path(".").resolve()
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
N_ACTAS = 5000   # cuantas actas preprocesar
SUBIR_A_HF = True  # publica crops_bundle.tar.gz en HF (requiere HF_TOKEN)

In [ ]:
# Plantilla Presidencial: 42 campos (38 partidos + blanco/nulos/impugnados + total).
# Cajas en fraccion [0,1]; embebida para que el notebook sea autonomo.
TEMPLATE = { "descripcion": "ACTA DE ESCRUTINIO PRESIDENCIAL (idEleccion=10, tipo=1). iter7 auto-detected.", "image_size_reference": [ 2339, 3309 ], "fields": [ { "name": "partido_01", "box": [ 0.462, 0.2149, 0.539, 0.2315 ], "n_digits": 3 }, { "name": "partido_02", "box": [ 0.462, 0.2315, 0.539, 0.2481 ], "n_digits": 3 }, { "name": "partido_03", "box": [ 0.462, 0.2481, 0.539, 0.2648 ], "n_digits": 3 }, { "name": "partido_04", "box": [ 0.462, 0.2648, 0.539, 0.2814 ], "n_digits": 3 }, { "name": "partido_05", "box": [ 0.462, 0.2814, 0.539, 0.298 ], "n_digits": 3 }, { "name": "partido_06", "box": [ 0.462, 0.298, 0.539, 0.3146 ], "n_digits": 3 }, { "name": "partido_07", "box": [ 0.462, 0.3146, 0.539, 0.3312 ], "n_digits": 3 }, { "name": "partido_08", "box": [ 0.462, 0.3312, 0.539, 0.3479 ], "n_digits": 3 }, { "name": "partido_09", "box": [ 0.462, 0.3479, 0.539, 0.3645 ], "n_digits": 3 }, { "name": "partido_10", "box": [ 0.462, 0.3645, 0.539, 0.3811 ], "n_digits": 3 }, { "name": "partido_11", "box": [ 0.462, 0.3811, 0.539, 0.3977 ], "n_digits": 3 }, { "name": "partido_12", "box": [ 0.462, 0.3977, 0.539, 0.4143 ], "n_digits": 3 }, { "name": "partido_13", "box": [ 0.462, 0.4143, 0.539, 0.431 ], "n_digits": 3 }, { "name": "partido_14", "box": [ 0.462, 0.431, 0.539, 0.4476 ], "n_digits": 3 }, { "name": "partido_15", "box": [ 0.462, 0.4476, 0.539, 0.4642 ], "n_digits": 3 }, { "name": "partido_16", "box": [ 0.462, 0.4642, 0.539, 0.4808 ], "n_digits": 3 }, { "name": "partido_17", "box": [ 0.462, 0.4808, 0.539, 0.4974 ], "n_digits": 3 }, { "name": "partido_18", "box": [ 0.462, 0.4974, 0.539, 0.5141 ], "n_digits": 3 }, { "name": "partido_19", "box": [ 0.462, 0.5141, 0.539, 0.5307 ], "n_digits": 3 }, { "name": "partido_20", "box": [ 0.462, 0.5307, 0.539, 0.5473 ], "n_digits": 3 }, { "name": "partido_21", "box": [ 0.462, 0.5473, 0.539, 0.5639 ], "n_digits": 3 }, { "name": "partido_22", "box": [ 0.462, 0.5639, 0.539, 0.5805 ], "n_digits": 3 }, { "name": "partido_23", "box": [ 0.462, 0.5805, 0.539, 0.5972 ], "n_digits": 3 }, { "name": "partido_24", "box": [ 0.462, 0.5972, 0.539, 0.6138 ], "n_digits": 3 }, { "name": "partido_25", "box": [ 0.462, 0.6138, 0.539, 0.6304 ], "n_digits": 3 }, { "name": "partido_26", "box": [ 0.462, 0.6304, 0.539, 0.647 ], "n_digits": 3 }, { "name": "partido_27", "box": [ 0.462, 0.647, 0.539, 0.6636 ], "n_digits": 3 }, { "name": "partido_28", "box": [ 0.462, 0.6636, 0.539, 0.6803 ], "n_digits": 3 }, { "name": "partido_29", "box": [ 0.462, 0.6803, 0.539, 0.6969 ], "n_digits": 3 }, { "name": "partido_30", "box": [ 0.462, 0.6969, 0.539, 0.7135 ], "n_digits": 3 }, { "name": "partido_31", "box": [ 0.462, 0.7135, 0.539, 0.7301 ], "n_digits": 3 }, { "name": "partido_32", "box": [ 0.462, 0.7301, 0.539, 0.7467 ], "n_digits": 3 }, { "name": "partido_33", "box": [ 0.462, 0.7467, 0.539, 0.7634 ], "n_digits": 3 }, { "name": "partido_34", "box": [ 0.462, 0.7634, 0.539, 0.78 ], "n_digits": 3 }, { "name": "partido_35", "box": [ 0.462, 0.78, 0.539, 0.7966 ], "n_digits": 3 }, { "name": "partido_36", "box": [ 0.462, 0.7966, 0.539, 0.8132 ], "n_digits": 3 }, { "name": "partido_37", "box": [ 0.462, 0.8132, 0.539, 0.8298 ], "n_digits": 3 }, { "name": "partido_38", "box": [ 0.462, 0.8298, 0.539, 0.8465 ], "n_digits": 3 }, { "name": "votos_blanco", "box": [ 0.462, 0.8465, 0.539, 0.8631 ], "n_digits": 3 }, { "name": "votos_nulos", "box": [ 0.462, 0.8631, 0.539, 0.8797 ], "n_digits": 3 }, { "name": "votos_impugnados", "box": [ 0.462, 0.8797, 0.539, 0.8963 ], "n_digits": 3 }, { "name": "total_ciudadanos", "box": [ 0.4363, 0.8963, 0.539, 0.913 ], "n_digits": 4 } ] }

## 1. PREPROCESAMIENTO — deteccion de digitos (editar aqui)

In [ ]:
# === PREPROCESAMIENTO: deteccion de digitos (EDITAR AQUI para cambiar el metodo) ===
# Metodo OFICIAL = zonal por plantilla: cada campo se recorta por su caja relativa
# y se parte en n_digits celdas equiespaciadas. La afin del template alinea bien
# en >98% de actas. Para cambiar "donde estan los digitos", reescribe crop_fields
# / split_digits (p.ej. projection profile, deteccion por contornos, un detector
# aprendido) manteniendo la firma: localizar -> {campo: [celda_0, celda_1, ...]}.
import fitz  # PyMuPDF
from PIL import Image

TARGET_SIZE = (2339, 3309)  # tamano fijo: PNGs uniformes, detector estable

def render_acta(pdf_path, out_dir):
    """PDF de acta -> PNG portrait de tamano fijo (primera pagina)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    with fitz.open(pdf_path) as doc:
        page = doc[0]
        if page.rect.width > page.rect.height:
            page.set_rotation(90)  # normaliza landscape -> portrait
        mat = fitz.Matrix(TARGET_SIZE[0] / page.rect.width,
                          TARGET_SIZE[1] / page.rect.height)
        out = out_dir / f"{Path(pdf_path).stem}_p0.png"
        page.get_pixmap(matrix=mat).save(out)
    return out

def crop_fields(image_path, template):
    """Recorta los 42 campos por sus cajas relativas [x0,y0,x1,y1] en [0,1]."""
    img = Image.open(image_path).convert("L")
    w, h = img.size
    out = {}
    for field in template["fields"]:
        x0, y0, x1, y1 = field["box"]
        out[field["name"]] = img.crop((int(x0 * w), int(y0 * h),
                                       int(x1 * w), int(y1 * h)))
    return out

def split_digits(field_img, n_digits):
    """Parte un campo en n_digits celdas equiespaciadas (izq -> der)."""
    w, h = field_img.size
    step = w / n_digits
    return [field_img.crop((int(i * step), 0, int((i + 1) * step), h))
            for i in range(n_digits)]

def localize_digits(image_path, template):
    """Localizador zonal: {campo: [celda_0, ...]}. Punto unico de 'donde estan'."""
    fields = crop_fields(image_path, template)
    return {f["name"]: split_digits(fields[f["name"]], f["n_digits"])
            for f in template["fields"]}

In [ ]:
# === Labels desde el ground truth ONPE + armado de crops/manifest ===
import csv
import pandas as pd

def es_celda_escrita(value, n_cells, pos):
    """Convencion ONPE: cifras right-justified, leading zeros en blanco.
    value=5,n=3 -> [vacio,vacio,'5']; value=0 -> todo vacio (nadie escribe '000')."""
    if value == 0:
        return False
    return pos >= n_cells - len(str(int(value)))

def int_to_digits(value, n_cells):
    """Entero -> lista de n_cells digitos right-justified. 18,3 -> [0,1,8]."""
    return [int(c) for c in str(int(value)).zfill(n_cells)]

def field_value_for(name, votos_acta, total_emitidos):
    """Entero del ground truth para un campo (partido / blanco-nulos-impugnados / total)."""
    if name.startswith("partido_"):
        pos = int(name.split("_")[1])
        row = votos_acta[votos_acta["nposicion"] == pos]
        return int(row.iloc[0]["nvotos"]) if len(row) else 0
    mapping = {"votos_blanco": 80, "votos_nulos": 81, "votos_impugnados": 82}
    if name in mapping:
        row = votos_acta[votos_acta["nposicion"] == mapping[name]]
        return int(row.iloc[0]["nvotos"]) if len(row) else 0
    if name == "total_ciudadanos":
        return int(total_emitidos)
    raise ValueError(name)

def build_crops_for_acta(png_path, archivo_id, id_acta, template,
                         votos, cabecera, crops_root, filtrar_vacias=True):
    """PNG + labels -> crops/<label>/<archivoId>_<campo>_c<pos>.png. Devuelve (guardados, filtrados)."""
    cab = cabecera[cabecera["idActa"] == id_acta]
    if len(cab) == 0 or pd.isna(cab.iloc[0]["totalVotosEmitidos"]):
        return 0, 0
    total = int(cab.iloc[0]["totalVotosEmitidos"])
    votos_acta = votos[votos["idActa"] == id_acta]
    cells = localize_digits(png_path, template)
    crops_root = Path(crops_root)
    n_saved = n_filt = 0
    for field in template["fields"]:
        name, n_cells = field["name"], field["n_digits"]
        value = field_value_for(name, votos_acta, total)
        labels = int_to_digits(value, n_cells)
        for pos, (label, dimg) in enumerate(zip(labels, cells[name])):
            if filtrar_vacias and not es_celda_escrita(value, n_cells, pos):
                n_filt += 1; continue
            d = crops_root / str(label); d.mkdir(parents=True, exist_ok=True)
            dimg.save(d / f"{archivo_id}_{name}_c{pos}.png")
            n_saved += 1
    return n_saved, n_filt

def build_manifest(crops_dir, out_csv):
    """crops/<label>/*.png -> manifest CSV (path,label) relativo a crops_dir."""
    crops_dir = Path(crops_dir); rows = []
    for label_dir in sorted(crops_dir.iterdir()):
        if label_dir.is_dir():
            for img in sorted(label_dir.glob("*.png")):
                rows.append((str(img.relative_to(crops_dir)), label_dir.name))
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["path", "label"]); w.writerows(rows)
    return len(rows)

## 2. Descargar PDFs + labels de Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download(HF_DATASET_REPO, repo_type="dataset", allow_patterns="labels/*",
                  local_dir=str(DATA))
archivos = pd.read_parquet(DATA / "labels/actas_archivos.parquet")
votos    = pd.read_parquet(DATA / "labels/actas_votos.parquet")
cabecera = pd.read_parquet(DATA / "labels/actas_cabecera.parquet")
pres = archivos[(archivos["tipo"] == 1) & (archivos["idEleccion"] == 10)]
ids = pres["archivoId"].tolist()
random.Random(42).shuffle(ids)
ids = ids[:N_ACTAS]
n = len(ids); ntr = int(n * 0.70); nva = int(n * 0.15)
splits = {"train": ids[:ntr], "val": ids[ntr:ntr + nva], "test": ids[ntr + nva:]}
print({k: len(v) for k, v in splits.items()})

pdf_dir = WORK / "pdfs"; pdf_dir.mkdir(exist_ok=True)
for i, aid in enumerate(ids):
    hf_hub_download(HF_DATASET_REPO, f"{aid}.pdf", repo_type="dataset", local_dir=str(pdf_dir))
    if (i + 1) % 200 == 0: print(f"  {i + 1}/{len(ids)} PDFs")
print("PDFs descargados")

## 3. Render + recorte + manifests por split

In [ ]:
aid_to_idacta = dict(zip(archivos["archivoId"], archivos["idActa"]))
# Restringir labels a las actas elegidas: acelera los joins por idActa del recorte.
sel_idactas = {int(aid_to_idacta[a]) for a in ids if a in aid_to_idacta}
votos    = votos[votos["idActa"].isin(sel_idactas)]
cabecera = cabecera[cabecera["idActa"].isin(sel_idactas)]
rendered = WORK / "rendered"
for split, sids in splits.items():
    croot = DATA / f"crops_{split}"
    saved = 0
    for aid in sids:
        pdf = pdf_dir / f"{aid}.pdf"
        if not pdf.exists(): continue
        png = render_acta(pdf, rendered)
        ns, _ = build_crops_for_acta(png, aid, int(aid_to_idacta[aid]),
                                     TEMPLATE, votos, cabecera, croot)
        saved += ns
    n_rows = build_manifest(croot, DATA / f"manifest_{split}.csv")
    print(f"{split}: {saved} crops, manifest {n_rows} filas")

## 4. Empaquetar y publicar el bundle de crops en HF

El entregable (`02_*`) baja este `crops_bundle.tar.gz` en `MODO="cache"`.

In [ ]:
bundle = WORK / "crops_bundle.tar.gz"
with tarfile.open(bundle, "w:gz") as t:
    for split in ("train", "val", "test"):
        t.add(DATA / f"crops_{split}", arcname=f"data/crops_{split}")
        t.add(DATA / f"manifest_{split}.csv", arcname=f"data/manifest_{split}.csv")
print("bundle:", bundle, round(bundle.stat().st_size / 1e6, 1), "MB")

if SUBIR_A_HF:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    api.upload_file(path_or_fileobj=str(bundle), path_in_repo="crops_bundle.tar.gz",
                    repo_id=HF_DATASET_REPO, repo_type="dataset")
    print("subido a", HF_DATASET_REPO)
else:
    print("SUBIR_A_HF=False: el bundle quedo local. Ponlo en True (con HF_TOKEN) para publicar.")

---
Listo. Con el bundle publicado, abre **`02_entregable_colab.ipynb`** (`MODO="cache"`) para entrenar y obtener las metricas finales.